# Relatório Técnico e Plano de Testes — Sistema de Gerenciamento de Vacinação

## 1. Explicação Técnica da Arquitetura

### 1.1 Modelagem de Dados em Memória
O sistema mantém o estado da aplicação durante toda a execução utilizando **listas de dicionários**, simulando entidades relacionais em memória sem dependência de banco de dados externo:
* **`unidades` (12 registros iniciais):** Armazena `id`, `nome`, `publico_alvo` e o saldo local `estoque` (inicializado em `0`). Inclui a `UBS Rural (Caso-Limite)` com `publico_alvo: 0` para validação de divisão por zero.
* **`lotes` (4 registros iniciais):** Armazena `id`, `vacina`, `quantidade_inicial`, `quantidade_disponivel` e `validade`. Separar a quantidade inicial da disponível permite calcular o percentual distribuído de cada lote sem perder o histórico original.
* **`aplicacoes`:** Registra cada evento de vacinação vinculando `unidade_id`, `unidade`, `quantidade` e `data` (`DD-MM-AAAA`), permitindo consolidar indicadores tanto por unidade quanto por dia.

### 1.2 Fluxo de Negócio e Integridade de Estoque
O controle de doses opera em **três etapas** (`Lote` $\rightarrow$ `Unidade` $\rightarrow$ `Aplicação`):
1. As doses entram no sistema centralizadas na lista `lotes`.
2. A função `abastecer_unidade()` valida o saldo do lote e transfere a quantidade solicitada para `unidade["estoque"]`.
3. A função `registrar_aplicacao()` consome exclusivamente do estoque local da unidade e registra a data da aplicação. Em todas as etapas, travas condicionais (`<= 0` e `> saldo`) impedem qualquer estoque negativo.

### 1.3 Modularização e Reaproveitamento de Código
* **`pedir_inteiro(mensagem)`:** Centraliza todas as entradas numéricas do sistema em um laço `while True` com tratamento de exceção (`try/except ValueError`), impedindo que entradas não numéricas interrompam o programa.
* **`encontrar_id(lista, id_alvo)`:** Função genérica de busca linear reutilizada em `abastecer_unidade()`, `registrar_aplicacao()`, `buscar_id()` e `editar_registro()`, evitando repetição de laços de busca.
* **Constante `ESTOQUE_CRITICO = 50`:** Define o limiar de alerta de doses críticas em um único ponto do código, facilitando a manutenção da regra na função `listar_criticos()`.

### 1.4 Arquitetura do Menu (Dicionário de Despacho)
O loop principal (`while True`) mapeia cada opção numérica diretamente para a referência da sua respectiva função por meio de um dicionário (`acoes = {"1": cadastrar_unidade, ...}`). O método `acoes.get(opcao)` valida opções inválidas e executa a função escolhida sem necessidade de encadear múltiplos blocos `if/elif`.

### 1.5 Consistência nas Atualizações e Casos-Limite
* **Prevenção de `ZeroDivisionError`:** Nos relatórios (`relatorio`, `relatorio_unidades` e `relatorio_lotes`), o cálculo percentual só é executado quando o divisor (`publico_alvo` ou `quantidade_inicial`) é maior que zero; caso contrário, o sistema exibe um aviso controlado.
* **Edição com consistência (`editar_registro`):** Ao atualizar um lote, o sistema impede que `quantidade_disponivel` seja negativa ou ultrapasse `quantidade_inicial`, preservando a coerência matemática dos relatórios consolidados.

---

## 2. Casos de Teste Documentados (8 Cenários)

### Teste 1 — Caso-Limite: Tentativa de aplicação em unidade sem estoque
* **Objetivo:** Validar se o sistema bloqueia a aplicação de doses quando a unidade ainda não foi abastecida (`estoque == 0`).
* **Entrada:** Opção `6` (Registrar aplicação) $\rightarrow$ ID da unidade: `1` (UBS Centro).
* **Saída Esperada:** O sistema recusa a operação antes de solicitar quantidade ou data, exibindo mensagem de falta de estoque e retornando ao menu.
* **Saída Observada:** `Unidade sem estoque disponivel.`

### Teste 2 — Caso Normal: Abastecimento de unidade a partir de um lote
* **Objetivo:** Validar a transferência de doses do estoque central (`lotes`) para o estoque local (`unidades`).
* **Entrada:** Opção `5` (Abastecer unidade) $\rightarrow$ ID da unidade: `1` (UBS Centro) $\rightarrow$ ID do lote: `1` (COVID-19, 400 doses) $\rightarrow$ Quantidade: `100`.
* **Saída Esperada:** Confirmação da transferência; Lote 1 atualizado de `400` para `300` doses disponíveis e UBS Centro atualizada de `0` para `100` doses em estoque.
* **Saída Observada:** `Unidade abastecida com sucesso!` (confirmado nas opções `2` e `4`).

### Teste 3 — Caso-Limite: Tentativa de abastecimento com lote zerado / estoque insuficiente
* **Objetivo:** Garantir que o sistema impeça saldo negativo nos lotes ao tentar transferir mais doses do que o disponível.
* **Entrada:** Opção `5` (Abastecer unidade) $\rightarrow$ ID da unidade: `2` (UBS Norte) $\rightarrow$ ID do lote: `4` (Febre Amarela, `0` doses disponíveis) $\rightarrow$ Quantidade: `10`.
* **Saída Esperada:** Bloqueio da transferência por saldo insuficiente no lote, mantendo ambos os estoques inalterados.
* **Saída Observada:** `Estoque insuficiente no lote.`

### Teste 4 — Caso Normal: Registro de aplicação de doses com controle por dia
* **Objetivo:** Validar a baixa no estoque da unidade e o armazenamento histórico com data na lista `aplicacoes`.
* **Entrada:** Opção `6` (Registrar aplicação) $\rightarrow$ ID da unidade: `1` (UBS Centro, estoque `100`) $\rightarrow$ Quantidade: `40` $\rightarrow$ Data: `19-11-2026`.
* **Saída Esperada:** Confirmação do registro; estoque da UBS Centro reduzido de `100` para `60` doses e registro gravado com a data `19-11-2026`.
* **Saída Observada:** `Aplicacao registrada com sucesso!`

### Teste 5 — Caso-Limite: Tentativa de aplicação maior que o estoque da unidade
* **Objetivo:** Garantir que uma unidade abastecida não registre aplicações acima do seu saldo disponível.
* **Entrada:** Opção `6` (Registrar aplicação) $\rightarrow$ ID da unidade: `1` (UBS Centro, agora com `60` doses) $\rightarrow$ Quantidade: `80`.
* **Saída Esperada:** Bloqueio da operação por estoque insuficiente na unidade, mantendo o saldo em `60` doses.
* **Saída Observada:** `Estoque insuficiente na unidade.`

### Teste 6 — Caso-Limite: Divisão por zero no cálculo de cobertura (Público-alvo zero)
* **Objetivo:** Verificar se o cálculo percentual de cobertura trata o divisor igual a zero sem lançar `ZeroDivisionError`.
* **Entrada:** Opção `8` (Relatório por unidade) — avaliando o registro ID `12` (`UBS Rural (Caso-Limite)`, com `publico_alvo: 0`).
* **Saída Esperada:** Exibição dos dados da unidade seguida de aviso controlado, sem interromper o loop do relatório.
* **Saída Observada:** `AVISO: Unidade sem publico-alvo cadastrado.`

### Teste 7 — Validação de Entrada: Digitação de texto em campo numérico
* **Objetivo:** Testar a resiliência da função auxiliar `pedir_inteiro()` contra entradas inválidas (`ValueError`).
* **Entrada:** Opção `1` (Cadastrar unidade) $\rightarrow$ Nome: `UBS Teste` $\rightarrow$ Público-alvo: `abc` $\rightarrow$ em seguida: `200`.
* **Saída Esperada:** Captura da exceção, exibição de alerta amigável e nova solicitação até receber um número inteiro válido.
* **Saída Observada:** `Digite apenas números inteiros!` seguido de `Unidade cadastrada com sucesso!` após digitar `200`.

### Teste 8 — Atualização com Consistência e Consolidação nos Indicadores
* **Objetivo:** Comprovar que a edição de registros respeita as regras de negócio e reflete dinamicamente nos relatórios.
* **Entrada:** Opção `12` (Editar registro) $\rightarrow$ `2` (Lote) $\rightarrow$ ID `1` $\rightarrow$ Nova quantidade: `500` (acima da inicial `400`) $\rightarrow$ Nova quantidade: `280` $\rightarrow$ seguida da Opção `7` (Relatório geral) e Opção `10` (Listar estoques críticos).
* **Saída Esperada:** Bloqueio da quantidade `500`; atualização para `280`; Relatório Geral exibindo `40` doses aplicadas, `19-11-2026: 40 doses` e saldos atualizados; Estoques Críticos destacando o Lote 3 (`35` doses, abaixo da constante `ESTOQUE_CRITICO = 50`).
* **Saída Observada:** `ERRO: Quantidade disponível não pode ser maior que a inicial (400)!`, seguido de `Lote alterado com sucesso!` e indicadores atualizados corretamente.